---
## Question 2: Scaling Up (Beryllium and Beyond)

The code you wrote for Helium works perfectly, but it is currently hardcoded for a 2-electron system where only one spatial orbital is filled ($N_{occ} = 1$). 

Let's scale our engine up to **Beryllium** ($Z=4$). Beryllium has 4 electrons. Because of the Pauli Exclusion Principle, we must fill two spatial orbitals ($1s$ and $2s$). Therefore, our occupation number is $N_{occ} = 2$.

The Roothaan-Hall equations do not change. The only thing that changes is how we build the total electron cloud (the Density Matrix, $\mathbf{P}$). Instead of just taking the first column of the Coefficient matrix $\mathbf{C}$, we must loop over the first *two* columns and sum them together:
$$P_{tu} = 2 \sum_{i=0}^{N_{occ}-1} C_{ti} C_{ui}$$

**Your Task:**
1. Modify the Beryllium SCF loop below. 
2. Change the nuclear charge to $Z=4$.
3. Locate **Step 8** and rewrite the `P_new` generation to use a `for` loop over `N_occ`, making your code universally applicable to any closed-shell atom!


In [ ]:
# We provide 4 optimized s-type Gaussians for Beryllium
alphas_Be = np.array([0.05, 0.2, 1.5, 6.0])
N = len(alphas_Be)

# YOUR CODE HERE: Set the new Nuclear Charge and Occupation Number
Z = 2      # Change this!
N_occ = 1  # Change this!

# Build Pre-Loop Matrices
S = np.zeros((N, N)); h_core = np.zeros((N, N)); V_ee = np.zeros((N, N, N, N))
for r in range(N):
    for s in range(N):
        a, b = alphas_Be[r], alphas_Be[s]
        S[r, s] = overlap(a, b)
        h_core[r, s] = kinetic(a, b) + nuclear(a, b, Z)
        for t in range(N):
            for u in range(N):
                c, d = alphas_Be[t], alphas_Be[u]
                V_ee[r, s, t, u] = repulsion(a, b, c, d)

X = fractional_matrix_power(S, -0.5)

# Generalized SCF Loop
P = np.zeros((N, N))
energy_old = 0.0

print("Iter | Electronic Energy (Hartrees) | Delta E")
print("-----------------------------------------------")

for iteration in range(25):
    # Step 4: Build Fock
    F = np.zeros((N, N))
    for r in range(N):
        for s in range(N):
            F[r, s] = h_core[r, s] 
            for t in range(N):
                for u in range(N):
                    F[r, s] += P[t, u] * (V_ee[r, s, t, u] - 0.5 * V_ee[r, u, t, s])
                    
    # Steps 5-7: Solve
    F_prime = np.dot(X, np.dot(F, X))
    E, C_prime = la.eigh(F_prime)
    C = np.dot(X, C_prime)
    
    # --- Step 8: MODIFY THIS TO HANDLE N_occ ---
    P_new = np.zeros((N, N))
    for t in range(N):
        for u in range(N):
            # Write a loop over i from 0 to N_occ
            # P_new[t, u] += 2 * C[t, i] * C[u, i]
            
            # Temporary hardcoded He formula (Replace this!):
            P_new[t, u] = 2 * C[t, 0] * C[u, 0] 
            
    # Step 9: Check Energy
    E_elec = 0.5 * np.sum(P_new * (h_core + F))
    delta_E = abs(E_elec - energy_old)
    print(f"{iteration+1:4d} | {E_elec:27.6f} | {delta_E:.2e}")
    
    if delta_E < 1e-6: break
    P = P_new; energy_old = E_elec

print(f"\nFinal Beryllium Energy: {E_elec:.6f} Hartrees")


---
### Entering the p-Block: Simulating Neon ($Z=10$)

Now we will test the true power of matrix algebra. We want to simulate **Neon**.
Neon has 10 electrons, which means it fills five spatial orbitals ($N_{occ} = 5$): the $1s, 2s, 2p_x, 2p_y$, and $2p_z$. 

Until now, we have only used $s$-type spherical Gaussians. A $p$-orbital has a "dumbbell" shape with a positive lobe and a negative lobe. Calculating the 3D integrals for $p$-orbitals analytically is a nightmare of calculus. 

However, **the computer doesn't care about the calculus.**
Once the Overlap ($\mathbf{S}$), Core ($\mathbf{h}_{core}$), and Repulsion ($\mathbf{V}_{ee}$) matrices are built, the Roothaan-Hall algorithm behaves exactly the same whether the basis functions are spheres, dumbbells, or donuts! The matrices just get bigger.

For a minimal representation of Neon, we need at least 5 basis functions. That means our matrices will be $5 \times 5$, and our 4D repulsion tensor will contain $5 \times 5 \times 5 \times 5 = 625$ elements!

**Your Task:**
1. I have provided a "black box" helper function `get_neon_matrices()` that instantly returns the pre-calculated integrals for Neon.
2. Copy and paste your *generalized* SCF loop from Beryllium into the cell below.
3. Update `N_occ` for Neon, feed it the new matrices, and run it!


In [ ]:
# --- HELPER FUNCTION (DO NOT MODIFY) ---
def get_neon_matrices():
    """Returns mock 5x5 integral matrices simulating Neon."""
    N_basis = 5
    S = np.eye(N_basis) # Simplified orthogonal basis
    h = np.diag([-30.0, -1.0, -0.5, -0.5, -0.5]) 
    V = np.zeros((N_basis, N_basis, N_basis, N_basis))
    
    # Adding some mock electron repulsion
    for i in range(N_basis):
        for j in range(N_basis):
            V[i,i,j,j] = 0.8  # Coulomb
            V[i,j,i,j] = 0.2  # Exchange
    return S, h, V
# ---------------------------------------

S_neon, h_neon, Vee_neon = get_neon_matrices()
Z_neon = 10
N_neon = len(S_neon)
X_neon = fractional_matrix_power(S_neon, -0.5)

# YOUR CODE HERE: Paste your generalized loop and change N_occ to solve Neon!




### Conceptual Checkpoint

**Question 2.1:** Look at the Density Matrix equation you wrote for Beryllium and Neon: $P_{tu} = 2 \sum_{i=0}^{N_{occ}-1} C_{ti} C_{ui}$. Notice that the $2$ at the front of the equation assumes every single occupied orbital has exactly two electrons in it. What type of atoms or molecules would cause this code to fundamentally break? *(Hint: Think about oxygen gas, or radicals).*

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 2.2:** In the Neon calculation, we needed a 4D array of size $5 \times 5 \times 5 \times 5 = 625$ elements just to store the repulsion integrals ($V_{ee}$) for a minimal basis set. If we simulated a drug molecule using a professional basis set with $1,000$ basis functions, how many elements would the $V_{ee}$ tensor contain? Why might RAM/Memory be a major bottleneck in computational chemistry?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**
